# HyRAG Replication Notebook

This notebook reproduces the public HyRAG pipeline using repository-relative paths. It implements sequential **semantic retrieval → rule-based retrieval → top-k semantic fallback**, followed by Mistral-7B template generation.

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import torch

from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForCausalLM
from unsloth import FastLanguageModel

# Make src/ importable whether the notebook is launched from the repo root or Scripts/.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "Scripts":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.hyrag import build_rules_dict, retrieve_context

## 1. Configuration

In [ ]:
DATASET_TO_RUN = "seen"  # "seen" or "unseen"
SIMILARITY_THRESHOLD = 0.7
TOP_K = 3
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_NEW_TOKENS = 64

DATA_DIR = REPO_ROOT / "Datasets"
RESULTS_DIR = REPO_ROOT / "Results"
RESULTS_DIR.mkdir(exist_ok=True)

TEST_FILE = DATA_DIR / ("Test_Seen_Dataset.csv" if DATASET_TO_RUN == "seen" else "Test_Unseen_Dataset.csv")
OUTPUT_FILE = RESULTS_DIR / f"HyRAG_Regenerated_{DATASET_TO_RUN.capitalize()}_Results.csv"

## 2. Load datasets

In [ ]:
context_df = pd.read_csv(DATA_DIR / "Context_Dataset.csv")
rules_df = pd.read_csv(DATA_DIR / "Rules_Dataset.csv")
test_df = pd.read_csv(TEST_FILE)

rules_dict = build_rules_dict(rules_df)

documents = [
    Document(page_content=f"Log: {row['Content']}\nTemplate: {row['EventTemplate']}")
    for _, row in context_df.iterrows()
]

print(f"Context rows: {len(context_df):,}")
print(f"Rule rows: {len(rules_df):,}")
print(f"Test rows: {len(test_df):,}")

## 3. Build semantic knowledge base

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)

def hyrag_retriever(log):
    return retrieve_context(
        log,
        embeddings=embeddings,
        vectorstore=vectorstore,
        rules_dict=rules_dict,
        k=TOP_K,
        threshold=SIMILARITY_THRESHOLD,
    )

## 4. Load compact LLM

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU is recommended/required for this 4-bit 7B model configuration.")

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to(device)
FastLanguageModel.for_inference(model)
EOS_TOKEN = tokenizer.eos_token or ""

## 5. Prompt and inference

In [ ]:
PROMPT = "You are a log parsing assistant. Your task is to extract the template of the given log message by replacing dynamic parts like timestamps, IDs, IP addresses, or numeric values with the '<*>' placeholder. Use the examples in the context to guide your output. If the log message is fully static and has no dynamic parts, return it as-is with no placeholders. Return ONLY the extracted template with no explanations, reasoning, or additional text.\n\n### Context:\n{selected_context}\n\n### Log Message to Parse:\n```\n{log}\n```\n\n### Response:\n"

def clean_generation(decoded_text):
    response = decoded_text.split("### Response:")[-1].strip()
    response = response.replace("`", "").strip()
    if EOS_TOKEN and EOS_TOKEN in response:
        response = response.split(EOS_TOKEN)[0].strip()
    if response.lower().startswith("template:"):
        response = response.split(":", 1)[1].strip()
    return response

def extract_log_templates(df):
    results = []
    for idx, row in enumerate(df.itertuples(index=False), start=1):
        log = str(getattr(row, "Content"))
        source = getattr(row, "Source", None)
        category = getattr(row, "Category", None)
        event_id = getattr(row, "EventId", None)
        ground_truth = getattr(row, "EventTemplate", None)

        selected_context, meta = hyrag_retriever(log)
        formatted_prompt = PROMPT.format(selected_context=selected_context, log=log) + EOS_TOKEN
        inputs = tokenizer([formatted_prompt], return_tensors="pt").to(device)

        start = time.time()
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, use_cache=True)
        elapsed = time.time() - start

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        extracted = clean_generation(decoded)

        results.append({
            "Content": log,
            "EventId": event_id,
            "EventTemplate": ground_truth,
            "Extracted Template": extracted,
            "Category": category,
            "Source": source,
            "Context_Source": meta["context_source"],
            "Matched_Rules": "|".join(meta.get("matched_rules", [])),
            "Similarity_Score": meta["similarity_score"],
            "Inference_Time_Seconds": elapsed,
        })

        if idx % 100 == 0 or idx == len(df):
            print(f"Processed {idx:,}/{len(df):,} logs")

    return pd.DataFrame(results)

## 6. Run and save results

In [ ]:
start = time.time()
results_df = extract_log_templates(test_df)
results_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")
print(f"Total runtime: {time.time() - start:.2f} seconds")
results_df.head()